# RAGLab — Phase 2: Experiment Analysis

This notebook analyses the results of the embedding model and chunking configuration experiments.

**Prerequisites:** run the experiments first:
```bash
python run_experiments.py --experiment all
```
Then start the MLflow UI (optional): `mlflow ui` → http://localhost:5000

---
**Metrics recap:**
| Metric | Type | Description |
|--------|------|-------------|
| `recall_at_k` | Retrieval | Fraction of expected sources found in top-k |
| `mrr` | Retrieval | Mean Reciprocal Rank of first relevant chunk |
| `precision_at_k` | Retrieval | Fraction of top-k chunks that are relevant |
| `faithfulness` | Generation | Answer grounded in context (1-5, LLM judge) |
| `answer_relevancy` | Generation | Answer addresses the question (1-5, LLM judge) |
| `context_precision` | Generation | Context relevant to question (1-5, LLM judge) |

In [ ]:
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import seaborn as sns

# Add project root to path so we can import src/
PROJECT_ROOT = Path("../").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

# Plot style
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120

RESULTS_DIR = PROJECT_ROOT / "data" / "eval_results"
print(f"Results directory: {RESULTS_DIR}")
print("Files found:", [f.name for f in sorted(RESULTS_DIR.glob("*.json"))] if RESULTS_DIR.exists() else "(none)")

## 1. Load MLflow Results

We load all experiment runs directly from MLflow's tracking store. This gives us a single DataFrame with parameters and metrics for every configuration tested.

In [ ]:
import mlflow

MLFLOW_URI = str(PROJECT_ROOT / "mlruns")
EXPERIMENT_NAME = "raglab_experiments"

mlflow.set_tracking_uri(MLFLOW_URI)

experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if experiment is None:
    print("⚠️  No MLflow experiment found. Run experiments first:")
    print("     python run_experiments.py --experiment all")
else:
    runs_df = mlflow.search_runs(
        experiment_ids=[experiment.experiment_id],
        order_by=["start_time ASC"],
    )
    print(f"Found {len(runs_df)} runs in experiment '{EXPERIMENT_NAME}'")
    runs_df.head()

In [ ]:
# Flatten: rename params.* and metrics.* columns for readability
param_cols  = [c for c in runs_df.columns if c.startswith("params.")]
metric_cols = [c for c in runs_df.columns if c.startswith("metrics.")]

df = runs_df[["tags.mlflow.runName"] + param_cols + metric_cols].copy()
df.columns = (
    ["run_name"]
    + [c.replace("params.", "") for c in param_cols]
    + [c.replace("metrics.", "") for c in metric_cols]
)

# Cast numeric columns
for col in ["chunk_size", "chunk_overlap", "top_k", "n_chunks"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

numeric_metrics = ["recall_at_k", "mrr", "precision_at_k",
                   "faithfulness", "answer_relevancy", "context_precision",
                   "index_time_s", "eval_time_s"]
for col in numeric_metrics:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Short model name for display
df["model_short"] = df["embedding_model"].str.split("/").str[-1]

print(df[["run_name", "embedding_model", "chunk_size", "chunk_overlap",
          "recall_at_k", "mrr", "precision_at_k"]].to_string(index=False))

## 2. Embedding Model Comparison

We compare three embedding models — `all-MiniLM-L6-v2` (baseline), `all-mpnet-base-v2`, and `BAAI/bge-small-en-v1.5` — across all three retrieval metrics. The chunking configuration is fixed at the baseline (512 chars / 64 overlap).

In [ ]:
# Filter to embedding experiments (fixed chunk config)
baseline_chunk_size = 512
baseline_overlap = 64
df_emb = df[
    (df["chunk_size"] == baseline_chunk_size) &
    (df["chunk_overlap"] == baseline_overlap)
].copy()

if df_emb.empty:
    print("No embedding experiment runs found — using all runs instead.")
    df_emb = df.copy()

metrics_to_plot = ["recall_at_k", "mrr", "precision_at_k"]

fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharey=False)
fig.suptitle("Retrieval Metrics by Embedding Model\n(chunk=512, overlap=64)",
             fontsize=13, fontweight="bold")

metric_labels = {"recall_at_k": "Recall@k", "mrr": "MRR",
                 "precision_at_k": "Precision@k"}
colors = sns.color_palette("muted", len(df_emb))

for ax, metric in zip(axes, metrics_to_plot):
    if metric not in df_emb.columns:
        ax.text(0.5, 0.5, f"{metric}\nnot available", ha="center", va="center",
                transform=ax.transAxes, fontsize=10, color="gray")
        continue
    bars = ax.bar(
        df_emb["model_short"], df_emb[metric],
        color=colors, edgecolor="white", linewidth=1.2
    )
    ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=9)
    ax.set_title(metric_labels[metric], fontweight="bold")
    ax.set_ylim(0, 1.15)
    ax.set_ylabel("Score")
    ax.tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "embedding_comparison.png", bbox_inches="tight")
plt.show()

### Observations — Embedding Models

*(Fill in after running experiments)*

- **`all-MiniLM-L6-v2`** is the fastest model (384-dim embeddings, ~80MB). It serves as our baseline.
- **`all-mpnet-base-v2`** is larger (768-dim, ~420MB) and typically achieves better recall due to richer representations — expect a 5–15% improvement in Recall@k.
- **`BAAI/bge-small-en-v1.5`** is optimised for retrieval tasks and often punches above its weight despite being small. Look for whether its retrieval-specific training outweighs the smaller dimension.

Key question: **Is the quality gain from a larger model worth the indexing and inference overhead?**

## 3. Chunking Configuration Comparison — Heatmap

We compare all six chunking configurations. The heatmap shows Recall@k for each `(chunk_size, chunk_overlap)` pair, revealing how granularity affects retrieval quality.

In [ ]:
# Filter to baseline embedding model
baseline_model = "all-MiniLM-L6-v2"
df_chunk = df[df["embedding_model"] == baseline_model].copy()

if df_chunk.empty:
    print(f"No runs found for model '{baseline_model}'. Using all runs.")
    df_chunk = df.copy()

# Pivot for heatmap
if "recall_at_k" in df_chunk.columns:
    pivot = df_chunk.pivot_table(
        index="chunk_size", columns="chunk_overlap", values="recall_at_k"
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"Chunking Config Results ({baseline_model})",
                 fontsize=13, fontweight="bold")

    # Left: Recall@k heatmap
    sns.heatmap(
        pivot, annot=True, fmt=".3f", cmap="YlGn",
        linewidths=0.5, ax=axes[0],
        cbar_kws={"label": "Recall@k"}
    )
    axes[0].set_title("Recall@k Heatmap", fontweight="bold")
    axes[0].set_xlabel("Chunk Overlap (chars)")
    axes[0].set_ylabel("Chunk Size (chars)")

    # Right: MRR heatmap
    if "mrr" in df_chunk.columns:
        pivot_mrr = df_chunk.pivot_table(
            index="chunk_size", columns="chunk_overlap", values="mrr"
        )
        sns.heatmap(
            pivot_mrr, annot=True, fmt=".3f", cmap="Blues",
            linewidths=0.5, ax=axes[1],
            cbar_kws={"label": "MRR"}
        )
        axes[1].set_title("MRR Heatmap", fontweight="bold")
        axes[1].set_xlabel("Chunk Overlap (chars)")
        axes[1].set_ylabel("Chunk Size (chars)")

    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "chunking_heatmap.png", bbox_inches="tight")
    plt.show()
else:
    print("recall_at_k not available — run experiments first.")

### Observations — Chunking

*(Fill in after running experiments)*

General patterns to look for:
- **Chunk size 256**: Produces more, smaller chunks → higher chance of a relevant chunk being in top-k (better recall), but each chunk has less context for generation.
- **Chunk size 1024**: Fewer, richer chunks → better context for generation but potentially lower recall if the relevant fact is buried in a large chunk.
- **Overlap**: Increasing overlap reduces the risk of a relevant passage being split across chunk boundaries, typically improving recall slightly at the cost of more stored chunks.

The sweet spot is often a medium chunk size (512) with moderate overlap (50–100 chars).

## 4. Indexing Speed vs. Retrieval Quality

A scatter plot comparing indexing time (proxy for computational cost) against Recall@k (quality). This trade-off is key for production decisions.

In [ ]:
if "index_time_s" in df.columns and "recall_at_k" in df.columns:
    fig, ax = plt.subplots(figsize=(9, 6))

    scatter = ax.scatter(
        df["index_time_s"], df["recall_at_k"],
        c=df["chunk_size"].astype(float), cmap="viridis",
        s=120, alpha=0.85, edgecolors="white", linewidth=0.8
    )
    plt.colorbar(scatter, ax=ax, label="Chunk Size (chars)")

    # Annotate each point with its model name
    for _, row in df.iterrows():
        ax.annotate(
            row["model_short"],
            xy=(row["index_time_s"], row["recall_at_k"]),
            xytext=(6, 4), textcoords="offset points",
            fontsize=8, alpha=0.8
        )

    ax.set_xlabel("Indexing Time (seconds)", fontsize=11)
    ax.set_ylabel("Recall@k", fontsize=11)
    ax.set_title("Trade-off: Indexing Speed vs Retrieval Quality",
                 fontsize=13, fontweight="bold")
    ax.set_ylim(0, 1.05)

    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "speed_vs_quality.png", bbox_inches="tight")
    plt.show()
else:
    print("index_time_s or recall_at_k not available — run experiments first.")

## 5. Performance by Difficulty Level

Breaking down metrics by question difficulty reveals where the RAG system struggles. Hard questions (cross-document reasoning) should score lower than easy ones (single-chunk answers).

In [ ]:
# Load per-question results from the most recent JSON file
result_files = sorted(RESULTS_DIR.glob("*.json")) if RESULTS_DIR.exists() else []

if result_files:
    # Use the most recent embeddings experiment or fallback to any file
    emb_files = [f for f in result_files if "embed" in f.name]
    target_file = emb_files[-1] if emb_files else result_files[-1]

    with open(target_file, encoding="utf-8") as f:
        raw = json.load(f)

    # Handle list of experiment results vs single result
    if isinstance(raw, list):
        # Take baseline (all-MiniLM-L6-v2) if available
        baseline_result = next(
            (r for r in raw if "MiniLM" in r.get("config", {}).get("embedding_model", "")),
            raw[0]
        )
    else:
        baseline_result = raw

    per_q = baseline_result.get("per_question", [])
    df_q = pd.DataFrame(per_q)

    if not df_q.empty:
        diff_order = ["easy", "medium", "hard"]
        df_diff = (
            df_q.groupby("difficulty")[["recall_at_k", "precision_at_k", "mrr"]]
            .mean()
            .reindex(diff_order)
        )

        ax = df_diff.plot(
            kind="bar", figsize=(9, 5), rot=0,
            color=sns.color_palette("muted", 3),
            edgecolor="white", linewidth=1.2
        )
        ax.set_title("Retrieval Metrics by Question Difficulty",
                     fontsize=13, fontweight="bold")
        ax.set_xlabel("Difficulty", fontsize=11)
        ax.set_ylabel("Score", fontsize=11)
        ax.set_ylim(0, 1.15)
        ax.legend(loc="upper right")
        for container in ax.containers:
            ax.bar_label(container, fmt="%.2f", padding=2, fontsize=8)
        plt.tight_layout()
        plt.savefig(RESULTS_DIR / "difficulty_breakdown.png", bbox_inches="tight")
        plt.show()
        print("\nMean metrics by difficulty:")
        print(df_diff.round(4))
    else:
        print("No per-question data found.")
else:
    print("No result files found. Run experiments first.")

### Observations — Difficulty Breakdown

A well-functioning RAG system should show:
- **Easy questions**: High Recall@k (≥ 0.8) — the answer is in a single chunk, retrieval is straightforward.
- **Medium questions**: Moderate Recall@k (0.5–0.8) — requires synthesising 2–3 sources.
- **Hard questions**: Lower Recall@k (0.3–0.6) — cross-document reasoning, the retriever must find the right chunks across multiple articles.

A large gap between easy and hard questions suggests the top-k setting may need increasing for complex queries, or that query expansion / re-ranking could help.

## 6. Summary Table — All Configurations

A complete table ranked by Recall@k, highlighting the best overall configuration.

In [ ]:
summary_cols = [
    "run_name", "model_short", "chunk_size", "chunk_overlap",
    "recall_at_k", "mrr", "precision_at_k", "index_time_s"
]
available_cols = [c for c in summary_cols if c in df.columns]

df_summary = df[available_cols].sort_values("recall_at_k", ascending=False)

# Highlight best row
def highlight_max(s):
    is_max = s == s.max()
    return ["background-color: #d4edda; font-weight: bold" if v else "" for v in is_max]

styled = (
    df_summary.style
    .apply(highlight_max, subset=["recall_at_k", "mrr", "precision_at_k"])
    .format({
        "recall_at_k": "{:.4f}",
        "mrr": "{:.4f}",
        "precision_at_k": "{:.4f}",
        "index_time_s": "{:.1f}s",
    }, na_rep="N/A")
    .set_caption("All Experiment Configurations — Sorted by Recall@k (best first)")
)
styled

## 7. Best Configuration & Conclusion

*(Update this section after running your experiments)*

In [ ]:
if "recall_at_k" in df.columns and not df["recall_at_k"].isna().all():
    best = df.loc[df["recall_at_k"].idxmax()]
    print("=" * 55)
    print("BEST CONFIGURATION")
    print("=" * 55)
    print(f"  Embedding model : {best.get('embedding_model', 'N/A')}")
    print(f"  Chunk size      : {best.get('chunk_size', 'N/A')} chars")
    print(f"  Chunk overlap   : {best.get('chunk_overlap', 'N/A')} chars")
    print("")
    print(f"  Recall@k        : {best.get('recall_at_k', 'N/A'):.4f}")
    print(f"  MRR             : {best.get('mrr', 'N/A'):.4f}")
    print(f"  Precision@k     : {best.get('precision_at_k', 'N/A'):.4f}")
    if pd.notna(best.get('faithfulness')):
        print(f"  Faithfulness    : {best.get('faithfulness', 'N/A'):.2f}/5")
        print(f"  Relevancy       : {best.get('answer_relevancy', 'N/A'):.2f}/5")
    print("=" * 55)
else:
    print("Run experiments to see the best configuration.")

### Conclusions

*(Fill in after running experiments)*

**Embedding model takeaway:** `[winner]` achieves the best Recall@k because [reason]. The trade-off vs `all-MiniLM-L6-v2` is [X]× slower indexing for [Y]% better retrieval — [worth it / not worth it for production].

**Chunking takeaway:** Chunk size [X] with overlap [Y] gives the best balance of retrieval recall and context richness. Smaller chunks improve recall (more chances to hit the right chunk) but provide less context to the LLM. Larger chunks help generation quality but may bury relevant facts.

**Recommendation for Phase 3 (API):** Deploy with `[best_model]`, chunk size `[X]`, overlap `[Y]`. This achieves Recall@[k] = [score] while keeping indexing time under [N] seconds per 1,000 chunks.

---
*This analysis would be strengthened in production by: (1) using a stronger LLM judge (GPT-4 or Claude) instead of Mistral for generation metrics; (2) a larger, human-annotated evaluation set; (3) testing on out-of-domain questions to check generalisation.*